In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import glob
import os
from scipy.stats import linregress

# ============================================================
# Settings
# ============================================================

EXP_INFO = {
    "esm-piControl": {
        "tas_path": "/nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Amon/tas/gn/latest",
        "tas_pattern": "tas_Amon_NorESM2-LM_esm-piControl_r1i1p1f1_gn_*.nc",
        "emission_file": None,
    },

    "esm-up2p0": {
        "tas_path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010",
        "tas_pattern": "tas_Amon_NorESM2-LM_esm-up2p0_r1i1p1f1_gn_*.nc",
        "emission_file": "/nird/datalake/NS2980K/projects/TipESM/inputdata/emissions/emissions-TIPMIP_CO2_anthro_surface_185001-209912_fv_1.9x2.5_esm-up2p0.nc",
    },

    "esm-up2p0-swl2p0": {
        "tas_path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009",
        "tas_pattern": "tas_Amon_NorESM2-LM_esm-up2p0-swl2p0_r1i1p1f1_gn_*.nc",
        "emission_file": "/nird/datalake/NS2980K/projects/TipESM/inputdata/emissions/emissions-TIPMIP_CO2_anthro_surface_195101-225012_fv_1.9x2.5_esm-up2p0-swl2p0.nc",
    },

    "esm-up2p0-swl4p0": {
        "tas_path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl4p0/v20251010",
        "tas_pattern": "tas_Amon_NorESM2-LM_esm-up2p0-swl4p0_r1i1p1f1_gn_*.nc",
        "emission_file": "/nird/datalake/NS2980K/projects/TipESM/inputdata/emissions/emissions-TIPMIP_CO2_anthro_surface_205701-235612_fv_1.9x2.5_esm-up2p0-swl4p0.nc",
    },
}

OUT_CSV = "tas_global_mean_linearity_metrics.csv"

REGRESS_X = "cumulative_emission"   # "annual_emission" or "cumulative_emission"
REF_NYEAR = 30


# ============================================================
# Functions
# ============================================================

def load_global_mean_tas(exp_path, pattern):
    files = sorted(glob.glob(os.path.join(exp_path, pattern)))

    if len(files) == 0:
        raise FileNotFoundError(f"No TAS files found: {os.path.join(exp_path, pattern)}")

    print(f"\nTAS files found: {len(files)}")
    print(f"First: {files[0]}")
    print(f"Last : {files[-1]}")

    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        use_cftime=True,
        parallel=False,
    )

    tas = ds["tas"]

    weights = np.cos(np.deg2rad(ds["lat"]))
    weights = xr.DataArray(
        weights,
        dims=("lat",),
        coords={"lat": ds["lat"]},
        name="lat_weight",
    )

    tas_gm = tas.weighted(weights).mean(dim=("lat", "lon"))
    tas_ann = tas_gm.groupby("time.year").mean("time")

    tas_ann.name = "tas_global_mean_annual"
    tas_ann.attrs["units"] = tas.attrs.get("units", "K")

    ds.close()

    return tas_ann


def make_regular_latlon_area(lat, lon):
    """
    Calculate grid-cell area for a regular lat-lon grid.
    """

    R = 6371000.0  # Earth radius [m]

    lat_vals = lat.values
    lon_vals = lon.values

    lat_b = np.zeros((lat_vals.size, 2))
    lon_b = np.zeros((lon_vals.size, 2))

    lat_mid = 0.5 * (lat_vals[:-1] + lat_vals[1:])
    lon_mid = 0.5 * (lon_vals[:-1] + lon_vals[1:])

    lat_b[1:, 0] = lat_mid
    lat_b[:-1, 1] = lat_mid
    lat_b[0, 0] = -90.0
    lat_b[-1, 1] = 90.0

    lon_b[1:, 0] = lon_mid
    lon_b[:-1, 1] = lon_mid
    lon_b[0, 0] = lon_vals[0] - 0.5 * (lon_vals[1] - lon_vals[0])
    lon_b[-1, 1] = lon_vals[-1] + 0.5 * (lon_vals[-1] - lon_vals[-2])

    lat_b_rad = np.deg2rad(lat_b)
    lon_b_rad = np.deg2rad(lon_b)

    dlon = lon_b_rad[:, 1] - lon_b_rad[:, 0]

    area_2d = (
        R**2
        * (np.sin(lat_b_rad[:, 1]) - np.sin(lat_b_rad[:, 0]))[:, None]
        * dlon[None, :]
    )

    area = xr.DataArray(
        area_2d,
        dims=("lat", "lon"),
        coords={"lat": lat, "lon": lon},
        name="area",
        attrs={"units": "m2"},
    )

    return area


def load_annual_global_emission_from_nc(emission_file):
    """
    Load annual globally integrated anthropogenic CO2 emission.

    Input variable:
        CO2_flux(time, lat, lon), units = kg m-2 s-1

    Output:
        pandas Series indexed by year, units = PgC yr-1

    Note:
        The file labels the flux as CO2_flux, but previous TIPMIP/AERA usage
        appears to treat this as carbon mass flux. Therefore conversion is:
        kg s-1 -> kg yr-1 -> PgC yr-1 by /1e12.
    """

    if emission_file is None:
        return None

    if not os.path.exists(emission_file):
        raise FileNotFoundError(f"No emission file found: {emission_file}")

    ds = xr.open_dataset(emission_file, use_cftime=True)

    if "CO2_flux" not in ds:
        raise KeyError(f"CO2_flux not found. Available variables: {list(ds.data_vars)}")

    co2 = ds["CO2_flux"]
    area = make_regular_latlon_area(ds["lat"], ds["lon"])

    sec_per_year = 365.0 * 24.0 * 3600.0

    co2_global_kg_s = (co2 * area).sum(dim=("lat", "lon"), skipna=True)
    co2_global_pg_yr_monthly = co2_global_kg_s * sec_per_year / 1.0e12

    emis_ann = co2_global_pg_yr_monthly.groupby("time.year").mean("time")
    emis_ann.name = "CO2_emission_PgC_yr"

    s = pd.Series(
        emis_ann.values.astype(float),
        index=emis_ann["year"].values.astype(int),
        name="CO2_emission_PgC_yr",
    )

    ds.close()

    return s


def to_first_nyear_anomaly(da, n_ref=30):
    """
    Convert annual DataArray to anomaly relative to first n_ref years.
    """

    ref = da.isel(year=slice(0, n_ref)).mean("year")
    anom = da - ref

    return pd.Series(
        anom.values.astype(float),
        index=anom["year"].values.astype(int),
        name=da.name,
    )


def calc_metrics(y, x=None):
    """
    Calculate trend, variability, and R2 against emission.

    y: pandas Series indexed by year
    x: pandas Series indexed by year, optional
    """

    y = y.dropna()

    years = y.index.values.astype(int)
    yval = y.values.astype(float)

    out = {
        "start_year": int(years.min()),
        "end_year": int(years.max()),
        "n_years": int(len(years)),
        "mean_anomaly": float(np.nanmean(yval)),
        "trend_per_year": np.nan,
        "trend_R2_time": np.nan,
        "std_total": float(np.nanstd(yval, ddof=1)),
        "std_detrended": np.nan,
        "emission_slope": np.nan,
        "R2_vs_emission": np.nan,
        "n_regression_years": 0,
    }

    if len(y) >= 3:
        res_time = linregress(years, yval)
        trend = res_time.slope
        fit = res_time.intercept + res_time.slope * years

        out["trend_per_year"] = trend
        out["trend_R2_time"] = res_time.rvalue**2
        out["std_detrended"] = float(np.nanstd(yval - fit, ddof=1))

    if x is not None:
        df = pd.concat([y.rename("y"), x.rename("x")], axis=1).dropna()

        if len(df) >= 3 and np.nanstd(df["x"].values) > 0:
            res_emis = linregress(df["x"].values, df["y"].values)

            out["emission_slope"] = res_emis.slope
            out["R2_vs_emission"] = res_emis.rvalue**2
            out["n_regression_years"] = int(len(df))

    return out


# ============================================================
# Main
# ============================================================

tas_ann_dict = {}
tas_anom_dict = {}
emission_dict = {}

for exp, info in EXP_INFO.items():
    print("\n======================================")
    print(exp)
    print("======================================")

    tas_ann = load_global_mean_tas(info["tas_path"], info["tas_pattern"])
    tas_ann_dict[exp] = tas_ann
    tas_anom_dict[exp] = to_first_nyear_anomaly(tas_ann, n_ref=REF_NYEAR)

    emis = load_annual_global_emission_from_nc(info["emission_file"])

    if emis is not None:
        if REGRESS_X == "annual_emission":
            emis_x = emis.copy()
            emis_x.name = "CO2_emission_PgC_yr"

        elif REGRESS_X == "cumulative_emission":
            emis_x = emis.cumsum()
            emis_x.name = "cumulative_CO2_emission_PgC"

        else:
            raise ValueError("REGRESS_X must be 'annual_emission' or 'cumulative_emission'.")

        emission_dict[exp] = emis_x

    else:
        emission_dict[exp] = None


rows = []

for exp in EXP_INFO:
    y = tas_anom_dict[exp]
    x = emission_dict[exp]

    metrics = calc_metrics(y, x=x)
    metrics["variable"] = "tas_global_mean"
    metrics["experiment"] = exp
    metrics["regression_x"] = REGRESS_X
    metrics["units"] = "K"

    rows.append(metrics)

table = pd.DataFrame(rows)


# ============================================================
# SNR
# ============================================================
# SNR is calculated only for esm-up2p0 here:
# signal = absolute linear forced change over the available experiment period
# noise  = detrended std from esm-piControl

picontrol_std = table.loc[
    table["experiment"] == "esm-piControl",
    "std_detrended"
].iloc[0]

table["SNR_vs_piControl"] = np.nan

for idx, row in table.iterrows():
    if row["experiment"] == "esm-up2p0":
        duration = row["end_year"] - row["start_year"] + 1
        forced_change = abs(row["trend_per_year"] * duration)
        table.loc[idx, "SNR_vs_piControl"] = forced_change / picontrol_std


# ============================================================
# Final table
# ============================================================

cols = [
    "variable",
    "experiment",
    "start_year",
    "end_year",
    "n_years",
    "units",
    "mean_anomaly",
    "trend_per_year",
    "std_total",
    "std_detrended",
    "SNR_vs_piControl",
    "regression_x",
    "emission_slope",
    "R2_vs_emission",
    "n_regression_years",
    "trend_R2_time",
]

table = table[cols]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

print("\nFinal table")
print(table)

table.to_csv(OUT_CSV, index=False)
print(f"\nSaved: {OUT_CSV}")


esm-piControl

TAS files found: 25
First: /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Amon/tas/gn/latest/tas_Amon_NorESM2-LM_esm-piControl_r1i1p1f1_gn_185101-186012.nc
Last : /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Amon/tas/gn/latest/tas_Amon_NorESM2-LM_esm-piControl_r1i1p1f1_gn_209101-210012.nc

esm-up2p0

TAS files found: 23
First: /nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010/tas_Amon_NorESM2-LM_esm-up2p0_r1i1p1f1_gn_185001-185912.nc
Last : /nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010/tas_Amon_NorESM2-LM_esm-up2p0_r1i1p1f1_gn_207001-207412.nc

esm-up2p0-swl2p0

TAS files found: 31
First: /nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009/tas_Amon_NorESM2-LM_esm-up2p0-swl2p0_r1i1p1f1_gn_195101-195912.nc
Last : /nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009/tas_Amon_NorESM2-LM_esm-up2p0-swl2p0_r1i1p1f1_gn_225001-225012.nc

esm-up2p0-swl4p0

TAS 

In [5]:
import xarray as xr
import numpy as np
import pandas as pd
import glob
import os
from scipy.stats import linregress


# ============================================================
# 1. Experiment information
# ============================================================

EXP_INFO = {
    "esm-piControl": {
        "path": "/nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Amon/tas/gn/latest",
        "pattern": "tas_Amon_NorESM2-LM_esm-piControl_r1i1p1f1_gn_*.nc",
    },
    "esm-up2p0": {
        "path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010",
        "pattern": "tas_Amon_NorESM2-LM_esm-up2p0_r1i1p1f1_gn_*.nc",
    },
    "esm-up2p0-swl2p0": {
        "path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009",
        "pattern": "tas_Amon_NorESM2-LM_esm-up2p0-swl2p0_r1i1p1f1_gn_*.nc",
    },
    "esm-up2p0-swl4p0": {
        "path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl4p0/v20251010",
        "pattern": "tas_Amon_NorESM2-LM_esm-up2p0-swl4p0_r1i1p1f1_gn_*.nc",
    },
}

VAR_NAME = "tas"
VAR_LABEL = "tas_global_mean"
REF_NYEAR = 30


# ============================================================
# 2. Load annual global mean TAS from files
# ============================================================

def load_annual_global_mean(var_name, exp_path, pattern):
    files = sorted(glob.glob(os.path.join(exp_path, pattern)))

    if len(files) == 0:
        raise FileNotFoundError(f"No files found: {os.path.join(exp_path, pattern)}")

    print("\nFiles found:", len(files))
    print("First:", files[0])
    print("Last :", files[-1])

    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        use_cftime=True,
        parallel=False,
    )

    da = ds[var_name]

    weights = np.cos(np.deg2rad(ds["lat"]))
    weights = xr.DataArray(
        weights,
        dims=("lat",),
        coords={"lat": ds["lat"]},
    )

    gm = da.weighted(weights).mean(dim=("lat", "lon"))
    ann = gm.groupby("time.year").mean("time")

    s = pd.Series(
        ann.values.astype(float),
        index=ann["year"].values.astype(int),
        name=var_name,
    )

    ds.close()

    return s


# ============================================================
# 3. Metric functions
# ============================================================

def make_anomaly(ts, n_ref=30):
    return ts - ts.iloc[:n_ref].mean()


def calc_detrended_std(ts):
    ts = ts.dropna()
    years = ts.index.values.astype(float)
    values = ts.values.astype(float)

    reg = linregress(years, values)
    fit = reg.intercept + reg.slope * years
    resid = values - fit

    return np.std(resid, ddof=1)


def calc_trend_metrics(ts):
    ts = ts.dropna()
    years = ts.index.values.astype(float)
    values = ts.values.astype(float)

    reg = linregress(years, values)

    duration = years[-1] - years[0]
    total_change = reg.slope * duration

    return {
        "trend_per_year": reg.slope,
        "trend_total_change": total_change,
        "trend_R2_time": reg.rvalue**2,
        "trend_pvalue": reg.pvalue,
    }


def calc_r2_vs_emission(ts, emission):
    df = pd.concat(
        [ts.rename("target"), emission.rename("emission")],
        axis=1,
    ).dropna()

    if len(df) < 3:
        return np.nan, np.nan, 0

    if df["emission"].std() == 0:
        return np.nan, np.nan, len(df)

    reg = linregress(df["emission"], df["target"])

    return reg.rvalue**2, reg.slope, len(df)


def evaluate_from_timeseries(
    variable_label,
    ts_dict,
    emission_dict,
    ref_nyear=30,
):
    """
    ts_dict should contain:
        esm-piControl
        esm-up2p0
        esm-up2p0-swl2p0
        esm-up2p0-swl4p0

    emission_dict should already contain:
        esm-up2p0
    """

    # Anomalies
    anom = {
        exp: make_anomaly(ts, n_ref=ref_nyear)
        for exp, ts in ts_dict.items()
    }

    # piControl internal variability
    sigma_pi = calc_detrended_std(anom["esm-piControl"])

    # up2p0 linearity against emission
    r2_up, slope_up, n_up = calc_r2_vs_emission(
        anom["esm-up2p0"],
        emission_dict["esm-up2p0"],
    )

    trend_up = calc_trend_metrics(anom["esm-up2p0"])
    snr_up = abs(trend_up["trend_total_change"]) / sigma_pi

    # zero-emission drift
    trend_sw2 = calc_trend_metrics(anom["esm-up2p0-swl2p0"])
    trend_sw4 = calc_trend_metrics(anom["esm-up2p0-swl4p0"])

    drift_sw2_sigma = trend_sw2["trend_total_change"] / sigma_pi
    drift_sw4_sigma = trend_sw4["trend_total_change"] / sigma_pi

    row = {
        "variable": variable_label,

        "up2p0_R2_vs_emission": r2_up,
        "up2p0_slope_vs_emission": slope_up,
        "up2p0_n_regression_years": n_up,
        "up2p0_SNR": snr_up,
        "up2p0_trend_per_year": trend_up["trend_per_year"],
        "up2p0_trend_total_change": trend_up["trend_total_change"],

        "swl2p0_drift_per_year": trend_sw2["trend_per_year"],
        "swl2p0_drift_total_change": trend_sw2["trend_total_change"],
        "swl2p0_drift_sigma": drift_sw2_sigma,
        "swl2p0_trend_R2_time": trend_sw2["trend_R2_time"],
        "swl2p0_trend_pvalue": trend_sw2["trend_pvalue"],

        "swl4p0_drift_per_year": trend_sw4["trend_per_year"],
        "swl4p0_drift_total_change": trend_sw4["trend_total_change"],
        "swl4p0_drift_sigma": drift_sw4_sigma,
        "swl4p0_trend_R2_time": trend_sw4["trend_R2_time"],
        "swl4p0_trend_pvalue": trend_sw4["trend_pvalue"],

        "piControl_detrended_std": sigma_pi,
    }

    return row


# ============================================================
# 4. Load TAS
# ============================================================

tas_ts_dict = {}

for exp, info in EXP_INFO.items():
    print("\n====================================")
    print(exp)
    print("====================================")

    tas_ts_dict[exp] = load_annual_global_mean(
        var_name=VAR_NAME,
        exp_path=info["path"],
        pattern=info["pattern"],
    )


# ============================================================
# 5. Evaluate TAS
# ============================================================
# Assumption:
# emission_dict["esm-up2p0"] already exists.
# It can be annual emission or cumulative emission.
# For TAS, cumulative emission is usually preferred.

row_tas = evaluate_from_timeseries(
    variable_label=VAR_LABEL,
    ts_dict=tas_ts_dict,
    emission_dict=emission_dict,
    ref_nyear=REF_NYEAR,
)

table = pd.DataFrame([row_tas])

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print(table)

table.to_csv("tas_metric_table.csv", index=False)
print("Saved: tas_metric_table.csv")


esm-piControl

Files found: 25
First: /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Amon/tas/gn/latest/tas_Amon_NorESM2-LM_esm-piControl_r1i1p1f1_gn_185101-186012.nc
Last : /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Amon/tas/gn/latest/tas_Amon_NorESM2-LM_esm-piControl_r1i1p1f1_gn_209101-210012.nc

esm-up2p0

Files found: 23
First: /nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010/tas_Amon_NorESM2-LM_esm-up2p0_r1i1p1f1_gn_185001-185912.nc
Last : /nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010/tas_Amon_NorESM2-LM_esm-up2p0_r1i1p1f1_gn_207001-207412.nc

esm-up2p0-swl2p0

Files found: 31
First: /nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009/tas_Amon_NorESM2-LM_esm-up2p0-swl2p0_r1i1p1f1_gn_195101-195912.nc
Last : /nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009/tas_Amon_NorESM2-LM_esm-up2p0-swl2p0_r1i1p1f1_gn_225001-225012.nc

esm-up2p0-swl4p0

Files found: 29


In [16]:
import xarray as xr
import numpy as np
import pandas as pd
import glob
import os
from scipy.stats import linregress
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ============================================================
# Settings
# ============================================================

EXPS = [
    "esm-piControl",
    "esm-up2p0",
    "esm-up2p0-swl2p0",
    "esm-up2p0-swl4p0",
]

REF_NYEAR = 30
OUT_CSV = "AERA_candidate_metric_table.csv"


# ============================================================
# Variable configs
# ============================================================

VAR_CONFIGS = {
    "tas_global_mean": {
        "loader": "tas_global_mean",
        "units": "K",
        "info": {
            "esm-piControl": {
                "path": "/nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Amon/tas/gn/latest",
                "pattern": "tas_Amon_NorESM2-LM_esm-piControl_r1i1p1f1_gn_*.nc",
                "var_name": "tas",
            },
            "esm-up2p0": {
                "path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010",
                "pattern": "tas_Amon_NorESM2-LM_esm-up2p0_r1i1p1f1_gn_*.nc",
                "var_name": "tas",
            },
            "esm-up2p0-swl2p0": {
                "path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009",
                "pattern": "tas_Amon_NorESM2-LM_esm-up2p0-swl2p0_r1i1p1f1_gn_*.nc",
                "var_name": "tas",
            },
            "esm-up2p0-swl4p0": {
                "path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl4p0/v20251010",
                "pattern": "tas_Amon_NorESM2-LM_esm-up2p0-swl4p0_r1i1p1f1_gn_*.nc",
                "var_name": "tas",
            },
        },
    },

    "omega_arag_surface_global_mean": {
        "loader": "omega_arag",
        "units": "1",
        "info": {
            "esm-piControl": {
                "path": "/nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon",
                "co3_pattern": "co3os/gr/latest/co3os_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_*.nc",
                "co3sat_pattern": "co3sataragos/gr/latest/co3sataragos_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_*.nc",
            },
            "esm-up2p0": {
                "path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010",
                "co3_pattern": "co3os_Omon_NorESM2-LM_esm-up2p0_r1i1p1f1_gr_*.nc",
                "co3sat_pattern": "co3sataragos_Omon_NorESM2-LM_esm-up2p0_r1i1p1f1_gr_*.nc",
            },
            "esm-up2p0-swl2p0": {
                "path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009",
                "co3_pattern": "co3os_Omon_NorESM2-LM_esm-up2p0-swl2p0_r1i1p1f1_gr_*.nc",
                "co3sat_pattern": "co3sataragos_Omon_NorESM2-LM_esm-up2p0-swl2p0_r1i1p1f1_gr_*.nc",
            },
            "esm-up2p0-swl4p0": {
                "path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl4p0/v20251010",
                "co3_pattern": "co3os_Omon_NorESM2-LM_esm-up2p0-swl4p0_r1i1p1f1_gr_*.nc",
                "co3sat_pattern": "co3sataragos_Omon_NorESM2-LM_esm-up2p0-swl4p0_r1i1p1f1_gr_*.nc",
            },
        },
    },

    "omz_volume_2mgL_global_sum": {
        "loader": "global_sum",
        "units": "m3",
        "info": {
            "esm-piControl": {
                "path": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-piControl/latest/omz_volume_2mgL",
                "pattern": "omzvol_2mgL_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_*.nc",
                "var_name": "omz_volume",
            },
            "esm-up2p0": {
                "path": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0/v20251010/omz_volume_2mgL",
                "pattern": "omzvol_2mgL_Omon_NorESM2-LM_esm-up2p0_r1i1p1f1_gr_*.nc",
                "var_name": "omz_volume",
            },
            "esm-up2p0-swl2p0": {
                "path": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl2p0/v20251009/omz_volume_2mgL",
                "pattern": "omzvol_2mgL_Omon_NorESM2-LM_esm-up2p0-swl2p0_r1i1p1f1_gr_*.nc",
                "var_name": "omz_volume",
            },
            "esm-up2p0-swl4p0": {
                "path": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl4p0/v20251010/omz_volume_2mgL",
                "pattern": "omzvol_2mgL_Omon_NorESM2-LM_esm-up2p0-swl4p0_r1i1p1f1_gr_*.nc",
                "var_name": "omz_volume",
            },
        },
    },

    "lowO2_volume_4p6mgL_global_sum": {
        "loader": "global_sum",
        "units": "m3",
        "info": {
            "esm-piControl": {
                "path": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-piControl/latest/lowO2_volume_4p6mgL",
                "pattern": "lowO2_volume_4p6mgL_NorESM2-LM_esm-piControl_r1i1p1f1_gr_*.nc",
                "var_name": "omz_volume",
            },
            "esm-up2p0": {
                "path": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0/v20251010/lowO2_volume_4p6mgL",
                "pattern": "lowO2vol_4p6mgL_Omon_NorESM2-LM_esm-up2p0_r1i1p1f1_gr_*.nc",
                "var_name": "low_o2_volume",
            },
            "esm-up2p0-swl2p0": {
                "path": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl2p0/v20251009/lowO2_volume_4p6mgL",
                "pattern": "lowO2_volume_4p6mgL_NorESM2-LM_esm-up2p0-swl2p0_r1i1p1f1_gr_*.nc",
                "var_name": "omz_volume",
            },
            "esm-up2p0-swl4p0": {
                "path": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl4p0/v20251010/lowO2_volume_4p6mgL",
                "pattern": "lowO2_volume_4p6mgL_NorESM2-LM_esm-up2p0-swl4p0_r1i1p1f1_gr_*.nc",
                "var_name": "omz_volume",
            },
        },
    },
}

# ============================================================
# Add O2 inventory metrics
# ============================================================

AREACELLO_FILE = (
    "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010/"
    "areacello_Ofx_NorESM2-LM_esm-up2p0_r1i1p1f1_gn.nc"
)

O2_INV_EXP_INFO = {
    "esm-piControl": {
        "path": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-piControl/latest/o2-inventories",
        "pattern": "o2-inventories_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_*.nc",
    },
    "esm-up2p0": {
        "path": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0/v20251010/o2-inventories",
        "pattern": "o2-inventories_Omon_NorESM2-LM_esm-up2p0_r1i1p1f1_gr_*.nc",
    },
    "esm-up2p0-swl2p0": {
        "path": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl2p0/v20251009/o2-inventories",
        "pattern": "o2-inventories_Omon_NorESM2-LM_esm-up2p0-swl2p0_r1i1p1f1_gr_*.nc",
    },
    "esm-up2p0-swl4p0": {
        "path": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl4p0/v20251010/o2-inventories",
        "pattern": "o2-inventories_Omon_NorESM2-LM_esm-up2p0-swl4p0_r1i1p1f1_gr_*.nc",
    },
}

O2_INV_VARS = [
    "o2_inv_mld",
    "o2_inv_100m",
    "o2_inv_200m",
    "o2_inv_300m",
    "o2_inv_500m",
    "o2_inv_700m",
    "o2_inv_1000m",
    "o2_inv_1500m",
    "o2_inv_2000m",
    "o2_inv_total",
]

for v in O2_INV_VARS:
    VAR_CONFIGS[f"{v}_global_sum"] = {
        "loader": "o2_inventory_global_sum",
        "units": "mol",
        "var_name": v,
        "areacello_file": AREACELLO_FILE,
        "info": {
            exp: {
                **O2_INV_EXP_INFO[exp],
                "var_name": v,
                "areacello_file": AREACELLO_FILE,
            }
            for exp in EXPS
        },
    }

# ============================================================
# Add surface O2 metric
# ============================================================

O2OS_EXP_INFO = {
    "esm-piControl": {
        "path": "/nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2os/gn/latest",
        "pattern": "o2os_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gn_*.nc",
        "var_name": "o2os",
        "areacello_file": AREACELLO_FILE,
    },
    "esm-up2p0": {
        "path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010",
        "pattern": "o2os_Omon_NorESM2-LM_esm-up2p0_r1i1p1f1_gn_*.nc",
        "var_name": "o2os",
        "areacello_file": AREACELLO_FILE,
    },
    "esm-up2p0-swl2p0": {
        "path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009",
        "pattern": "o2os_Omon_NorESM2-LM_esm-up2p0-swl2p0_r1i1p1f1_gn_*.nc",
        "var_name": "o2os",
        "areacello_file": AREACELLO_FILE,
    },
    "esm-up2p0-swl4p0": {
        "path": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl4p0/v20251010",
        "pattern": "o2os_Omon_NorESM2-LM_esm-up2p0-swl4p0_r1i1p1f1_gn_*.nc",
        "var_name": "o2os",
        "areacello_file": AREACELLO_FILE,
    },
}

VAR_CONFIGS["o2os_surface_global_mean"] = {
    "loader": "o2os_global_mean",
    "units": "mg L-1",
    "info": O2OS_EXP_INFO,
}

# ============================================================
# Loader functions
# ============================================================

def _open_files(exp_path, pattern):
    files = sorted(glob.glob(os.path.join(exp_path, pattern)))

    if len(files) == 0:
        raise FileNotFoundError(f"No files found: {os.path.join(exp_path, pattern)}")

    print(f"Found {len(files)} files")
    print("First:", files[0])
    print("Last :", files[-1])

    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        use_cftime=True,
        data_vars="minimal",
        coords="minimal",
        compat="override",
        parallel=False,
    )

    return ds


def _to_series(da):
    return pd.Series(
        da.values.astype(float),
        index=da["year"].values.astype(int),
        name=da.name,
    )


def load_tas_global_mean(info):
    ds = _open_files(info["path"], info["pattern"])

    da = ds[info["var_name"]]

    weights = np.cos(np.deg2rad(ds["lat"]))
    weights = xr.DataArray(weights, dims=("lat",), coords={"lat": ds["lat"]})

    gm = da.weighted(weights).mean(dim=("lat", "lon"))
    ann = gm.groupby("time.year").mean("time")
    ann.name = "tas_global_mean_annual"

    s = _to_series(ann)

    ds.close()
    return s


def load_global_sum(info):
    ds = _open_files(info["path"], info["pattern"])

    da = ds[info["var_name"]]

    fill_value = da.attrs.get("_FillValue", None)
    if fill_value is not None:
        da = da.where(da != fill_value)

    spatial_dims = [d for d in ["j", "i", "lat", "lon"] if d in da.dims]

    gs = da.sum(dim=spatial_dims, skipna=True)
    ann = gs.groupby("time.year").mean("time")
    ann.name = f"{info['var_name']}_global_sum_annual"

    s = _to_series(ann)

    ds.close()
    return s


def load_omega_arag(info):
    co3_files = sorted(glob.glob(os.path.join(info["path"], info["co3_pattern"])))
    sat_files = sorted(glob.glob(os.path.join(info["path"], info["co3sat_pattern"])))

    if len(co3_files) == 0:
        raise FileNotFoundError(f"No co3os files found: {os.path.join(info['path'], info['co3_pattern'])}")
    if len(sat_files) == 0:
        raise FileNotFoundError(f"No co3sataragos files found: {os.path.join(info['path'], info['co3sat_pattern'])}")

    print(f"co3os files        : {len(co3_files)}")
    print(f"co3sataragos files : {len(sat_files)}")
    print("co3 first:", co3_files[0])
    print("co3 last :", co3_files[-1])
    print("sat first:", sat_files[0])
    print("sat last :", sat_files[-1])

    ds_co3 = xr.open_mfdataset(
        co3_files,
        combine="by_coords",
        use_cftime=True,
        data_vars="minimal",
        coords="minimal",
        compat="override",
        chunks={"time": -1},
    )

    ds_sat = xr.open_mfdataset(
        sat_files,
        combine="by_coords",
        use_cftime=True,
        data_vars="minimal",
        coords="minimal",
        compat="override",
        chunks={"time": -1},
    )

    omega = ds_co3["co3os"] / ds_sat["co3sataragos"]
    omega.name = "omega_arag"

    weights = np.cos(np.deg2rad(ds_co3["latitude"]))
    weights = weights.where(np.isfinite(omega.isel(time=0)), 0.0).fillna(0.0)

    gm = omega.weighted(weights).mean(dim=("j", "i"), skipna=True).compute()
    ann = gm.groupby("time.year").mean("time")
    ann.name = "omega_arag_surface_global_mean_annual"

    s = _to_series(ann)

    ds_co3.close()
    ds_sat.close()

    return s


def load_variable_timeseries(var_cfg, exp):
    loader = var_cfg["loader"]
    info = var_cfg["info"][exp]

    if loader == "tas_global_mean":
        return load_tas_global_mean(info)

    if loader == "omega_arag":
        return load_omega_arag(info)

    if loader == "global_sum":
        return load_global_sum(info)

    if loader == "o2_inventory_global_sum":
        return load_o2_inventory_global_sum(info)

    if loader == "o2os_global_mean":
        return load_o2os_global_mean(info)

    raise ValueError(f"Unknown loader: {loader}")

def load_o2_inventory_global_sum(info):
    ds = _open_files(info["path"], info["pattern"])

    ds_area = xr.open_dataset(info["areacello_file"])
    area = ds_area["areacello"]

    var_name = info["var_name"]
    da = ds[var_name]

    # mol m-2 * m2 = mol
    global_sum = (da * area).sum(dim=("j", "i"), skipna=True)

    ann = global_sum.groupby("time.year").mean("time")
    ann.name = f"{var_name}_global_sum_annual"

    s = _to_series(ann)

    ds.close()
    ds_area.close()

    return s

def load_o2os_global_mean(info):
    O2_MOLAR_MASS = 31.998  # mol m-3 -> mg L-1

    ds = _open_files(info["path"], info["pattern"])

    ds_area = xr.open_dataset(info["areacello_file"])
    area = ds_area["areacello"]

    # Remove possible non-spatial dimensions
    for dim in list(area.dims):
        if dim not in ["j", "i"]:
            area = area.isel({dim: 0})

    area = area.squeeze(drop=True)
    area = area.where(np.isfinite(area))

    o2os = ds[info["var_name"]]

    # mol m-3 -> mg L-1
    # 1 mol O2 m-3 = 31.998 mg L-1
    o2os_mg_L = o2os * O2_MOLAR_MASS
    o2os_mg_L.attrs["units"] = "mg L-1"

    # Mask area using valid ocean values
    area_valid = area.where(np.isfinite(o2os_mg_L.isel(time=0)))

    gm = (
        (o2os_mg_L * area_valid).sum(dim=("j", "i"), skipna=True)
        / area_valid.sum(dim=("j", "i"), skipna=True)
    )

    ann = gm.groupby("time.year").mean("time")
    ann.name = "o2os_surface_global_mean_annual"

    s = _to_series(ann)

    ds.close()
    ds_area.close()

    return s

# ============================================================
# Metric functions
# ============================================================

def make_anomaly(ts, n_ref=30):
    return ts - ts.iloc[:n_ref].mean()


def calc_trend(ts):
    ts = ts.dropna()
    years = ts.index.values.astype(float)
    values = ts.values.astype(float)

    reg = linregress(years, values)
    duration = years[-1] - years[0]

    return {
        "trend_per_year": reg.slope,
        "trend_total_change": reg.slope * duration,
        "trend_R2_time": reg.rvalue**2,
        "trend_pvalue": reg.pvalue,
    }


def calc_detrended_std(ts):
    trend = calc_trend(ts)

    ts = ts.dropna()
    years = ts.index.values.astype(float)
    values = ts.values.astype(float)

    fit = trend["trend_per_year"] * years
    fit = fit + (values.mean() - fit.mean())

    resid = values - fit

    return float(np.std(resid, ddof=1))


def calc_r2_vs_emission(ts, emission):
    df = pd.concat(
        [ts.rename("target"), emission.rename("emission")],
        axis=1,
    ).dropna()

    if len(df) < 3:
        return np.nan, np.nan, 0

    if df["emission"].std() == 0:
        return np.nan, np.nan, len(df)

    reg = linregress(df["emission"], df["target"])

    return reg.rvalue**2, reg.slope, len(df)


def evaluate_variable(variable_label, units, ts_dict, emission_dict, ref_nyear=30):
    anom = {
        exp: make_anomaly(ts_dict[exp], n_ref=ref_nyear)
        for exp in EXPS
    }

    sigma_pi = calc_detrended_std(anom["esm-piControl"])

    r2_up, slope_up, n_up = calc_r2_vs_emission(
        anom["esm-up2p0"],
        emission_dict["esm-up2p0"],
    )

    trend_pi = calc_trend(anom["esm-piControl"])
    trend_up = calc_trend(anom["esm-up2p0"])
    trend_sw2 = calc_trend(anom["esm-up2p0-swl2p0"])
    trend_sw4 = calc_trend(anom["esm-up2p0-swl4p0"])

    row = {
        "variable": variable_label,
        "units": units,

        "piControl_detrended_std": sigma_pi,
        "piControl_drift_per_year": trend_pi["trend_per_year"],
        "piControl_drift_total_change": trend_pi["trend_total_change"],
        "piControl_drift_sigma": trend_pi["trend_total_change"] / sigma_pi,
        "piControl_drift_abs_sigma": abs(trend_pi["trend_total_change"]) / sigma_pi,
        "piControl_trend_R2_time": trend_pi["trend_R2_time"],
        "piControl_trend_pvalue": trend_pi["trend_pvalue"],

        "up2p0_R2_vs_emission": r2_up,
        "up2p0_slope_vs_emission": slope_up,
        "up2p0_n_regression_years": n_up,
        "up2p0_SNR": abs(trend_up["trend_total_change"]) / sigma_pi,
        "up2p0_trend_per_year": trend_up["trend_per_year"],
        "up2p0_trend_total_change": trend_up["trend_total_change"],

        "swl2p0_drift_per_year": trend_sw2["trend_per_year"],
        "swl2p0_drift_total_change": trend_sw2["trend_total_change"],
        "swl2p0_drift_sigma": trend_sw2["trend_total_change"] / sigma_pi,
        "swl2p0_drift_abs_sigma": abs(trend_sw2["trend_total_change"]) / sigma_pi,
        "swl2p0_trend_R2_time": trend_sw2["trend_R2_time"],
        "swl2p0_trend_pvalue": trend_sw2["trend_pvalue"],

        "swl4p0_drift_per_year": trend_sw4["trend_per_year"],
        "swl4p0_drift_total_change": trend_sw4["trend_total_change"],
        "swl4p0_drift_sigma": trend_sw4["trend_total_change"] / sigma_pi,
        "swl4p0_drift_abs_sigma": abs(trend_sw4["trend_total_change"]) / sigma_pi,
        "swl4p0_trend_R2_time": trend_sw4["trend_R2_time"],
        "swl4p0_trend_pvalue": trend_sw4["trend_pvalue"],
    }

    return row


In [17]:
# ============================================================
# Main loop with resume / skip existing variables
# ============================================================

if os.path.exists(OUT_CSV):
    table_old = pd.read_csv(OUT_CSV)
    done_vars = set(table_old["variable"].values)
    rows = table_old.to_dict("records")
    print(f"Loaded existing table: {OUT_CSV}")
    print(f"Already completed variables: {len(done_vars)}")
else:
    done_vars = set()
    rows = []

for var_label, var_cfg in VAR_CONFIGS.items():

    if var_label in done_vars:
        print(f"\nSkipping already completed variable: {var_label}")
        continue

    print("\n##################################################")
    print(f"Processing variable: {var_label}")
    print("##################################################")

    ts_dict = {}

    try:
        for exp in EXPS:
            print("\n======================================")
            print(exp)
            print("======================================")

            ts_dict[exp] = load_variable_timeseries(var_cfg, exp)

        row = evaluate_variable(
            variable_label=var_label,
            units=var_cfg["units"],
            ts_dict=ts_dict,
            emission_dict=emission_dict,
            ref_nyear=REF_NYEAR,
        )

        rows.append(row)

        # Save after each variable
        table = pd.DataFrame(rows)
        table.to_csv(OUT_CSV, index=False)

        print(f"Saved updated table after {var_label}: {OUT_CSV}")

    except Exception as e:
        print(f"\nFAILED: {var_label}")
        print(e)
        continue

table = pd.DataFrame(rows)

print("\nFinal metric table")
print(table)

Loaded existing table: AERA_candidate_metric_table.csv
Already completed variables: 14

Skipping already completed variable: tas_global_mean

Skipping already completed variable: omega_arag_surface_global_mean

Skipping already completed variable: omz_volume_2mgL_global_sum

Skipping already completed variable: lowO2_volume_4p6mgL_global_sum

Skipping already completed variable: o2_inv_mld_global_sum

Skipping already completed variable: o2_inv_100m_global_sum

Skipping already completed variable: o2_inv_200m_global_sum

Skipping already completed variable: o2_inv_300m_global_sum

Skipping already completed variable: o2_inv_500m_global_sum

Skipping already completed variable: o2_inv_700m_global_sum

Skipping already completed variable: o2_inv_1000m_global_sum

Skipping already completed variable: o2_inv_1500m_global_sum

Skipping already completed variable: o2_inv_2000m_global_sum

Skipping already completed variable: o2_inv_total_global_sum

##########################################

In [10]:
OUT_CSV

'AERA_candidate_metric_table.csv'